In [3]:
import torch.nn as nn
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
device = torch.device("cpu") #bc i'm doing this in aws, no gpu needed
transform = transforms.ToTensor() #this is a torchvision package for casting images to torch.Tensor
test_ds = datasets.MNIST(root="", train=False, download=True, transform=transform)
test_loader = DataLoader(test_ds, batch_size=56, shuffle=False)
model = nn.Sequential(
    nn.Flatten(), #unsurprisingly, this flattens data 0
    nn.Linear(28*28, 256), #linear layer 1
    nn.ReLU(), # 2
    nn.Linear(256, 128), #squash down 3
    nn.ReLU(), # 4
    nn.Linear(128, 10) #output 5
).to(device)
model.load_state_dict(torch.load("mlp_mnist_state_dict.pt", map_location=device))
model.eval()
model.eval()
n1 = model[1].out_features
n2 = model[3].out_features
n3 = model[5].out_features #just doing this so i can make a concept mask
print(n1, n2, n3)

256 128 10


In [4]:
acts = {}#it's safe to put this outside of the function as any changes to it will be made in the function

def save_activation(name):
    def hook(module, inp, out):
        acts[name] = out.detach()
    return hook

def get_activations_for_digit(digit, n1=256, n2=128, n3=10):

    sum_loop_1 = torch.zeros(n1, device=device)
    sum_noloop_1 = torch.zeros(n1, device=device)

    sum_loop_2 = torch.zeros(n2, device=device)
    sum_noloop_2 = torch.zeros(n2, device=device)

    sum_loop_3 = torch.zeros(n3, device=device)
    sum_noloop_3 = torch.zeros(n3, device=device)

    cnt_loop = 0
    cnt_noloop = 0 

    h1 = model[2].register_forward_hook(save_activation("a1"))
    h2 = model[4].register_forward_hook(save_activation("a2"))

    with torch.no_grad():
        for x, y in test_loader:
            x = x.to(device)#x is a  B*28*28 or w/e
            y = y.to(device)#y is just B

            is_loop = torch.zeros_like(y, dtype=torch.bool)#y has dimensions of y which has same dimensions as batch
            
            is_loop |= (y == digit)

            output = model(x)

            a1 = acts.pop("a1")
            a2 = acts.pop("a2")

            if is_loop.any():
                sum_loop_1 += a1[is_loop].sum(dim=0)#a1 has dimensions B, n1. a B dimension mask can be applied to it
                sum_loop_2 += a2[is_loop].sum(dim=0)#gets sum of activations for loopy numbers
                sum_loop_3 += output[is_loop].sum(dim=0)
                cnt_loop += int(is_loop.sum().item()) 

            if (~is_loop).any():
                sum_noloop_1 += a1[~is_loop].sum(dim=0)
                sum_noloop_2 += a2[~is_loop].sum(dim=0)
                sum_noloop_3 += output[~is_loop].sum(dim=0)
                cnt_noloop += int((~is_loop).sum().item())

        h1.remove()
        h2.remove()

        div_val = max(1, cnt_loop)#pointless optimisation is fun
        
        mean_loop_1 = sum_loop_1 / div_val
        mean_loop_2 = sum_loop_2 / div_val
        mean_loop_3 = sum_loop_3 / div_val

        div_val = max(1, cnt_noloop)
        mean_noloop_1 = sum_noloop_1 / div_val
        mean_noloop_2 = sum_noloop_2 / div_val
        mean_noloop_3 = sum_noloop_3 / div_val

        loop_activations = [mean_loop_1, mean_loop_2, mean_loop_3]
        noloop_activations = [mean_noloop_1, mean_noloop_2, mean_noloop_3]
        #if i remember correctly, i can calculate the scores here

        return loop_activations, noloop_activations

In [5]:
#calculate mean activations on l1
digit_l1_means = {}

for d in range(10):
    acts_d, _ = get_activations_for_digit(d)
    digit_l1_means[d] = acts_d[1]

In [2]:
loop_digits = [0, 6, 8, 9]#loopy digits
nonloop_digits = [1, 2, 3, 4, 5, 7]#guess what these are

loop_mean = torch.stack([digit_l1_means[d] for d in loop_digits]).mean(dim=0)#dim=0 to get mean across values as opposed to keys
nonloop_mean = torch.stack([digit_l1_means[d] for d in nonloop_digits]).mean(dim=0)

score_loop = loop_mean - nonloop_mean#calculating mean of l1 loopy activations - mean of l1 non loopy activation 
top_loopy_neurons = torch.topk(score_loop, k=40).indices.tolist()#get top 40 in loop mean - non loop mean

residuals = {d: digit_l1_means[d] - loop_mean for d in loop_digits}#calculate residuals by iterating through l1 activations for all loopy digits and then subtracting mean of loops
top_residuals = {
    d: torch.topk(residuals[d], k=40).indices.tolist() for d in loop_digits
}#this provides the specific activations for each loopy digit

print("shared loop neurons:", top_loopy_neurons)#this shows top neuron activations for loopy digits when the mean for all loopy digits is subtracted for each loopy digit 

for d in loop_digits:
    print(f"top 40 digit-specific residual neurons for {d}:", top_residuals[d])

NameError: name 'torch' is not defined

In [7]:
#i now have a chance to test my hypothesis about eight being loop feature + double activation. if my hypothesis is true, 8 and 0 should have similar neuron activations
set_8 = set(top_residuals[8])
set_0 = set(top_residuals[0])
set_6 = set(top_residuals[6])
set_9 = set(top_residuals[9])

set8_0 = set_8 & set_0
set8_6 = set_8 & set_6
set8_9 = set_8 & set_9

set0_6 = set_0 & set_6
set0_8 = set_0 & set_8
set0_9 = set_0 & set_9

In [8]:
#they do have common neurons but does this mean anything? i can test this in two ways. 1) number of shared neurons 2) activation of shared neurons. 
print(set0_9)
print(set0_6)
print(set0_8)
#0 seems to have more shared activations with 6 than 8

{34, 8, 44, 14, 54, 122}
{64, 69, 70, 15, 48, 22, 24, 58, 27, 62, 95}
{122, 83, 53, 86}


In [9]:
#8 seems to have more shared activations with 9 than 6
print(set8_9)
print(set8_6)
print(set8_0)

{96, 1, 66, 98, 99, 41, 74, 12, 77, 109, 49, 87, 55, 122, 28, 30}
{4, 36, 71, 72, 10, 42, 47, 49, 23, 57, 90, 126}
{122, 83, 53, 86}


In [10]:
import torch.nn.functional as F


def similarity_on_set(vec_a, vec_b, neuron_set):
    idx = torch.tensor(sorted(neuron_set), device=vec_a.device)
    a = vec_a[idx]
    b = vec_b[idx]
    return {
        "cosine_similarity": F.cosine_similarity(a, b, dim=0).item(),
        "rms_diff": (a - b).pow(2).mean().sqrt().item(),
    }


print(similarity_on_set(residuals[8], residuals[0], set8_0))
print(similarity_on_set(residuals[8], residuals[9], set8_9))
print(similarity_on_set(residuals[8], residuals[6], set8_6))

{'cosine_similarity': 0.9296437501907349, 'rms_diff': 0.9510633945465088}
{'cosine_similarity': 0.8324816226959229, 'rms_diff': 0.7182475924491882}
{'cosine_similarity': 0.8190291523933411, 'rms_diff': 0.7501765489578247}


In [11]:
#what does above actually prove? 
#cosine simularity is calculated by 
#dot(a, b) / (mag(a) * mag(b)) where len(a) = len(b)
#magnitude of a vector is square every component of the vector, sum the squares, and then take the root
#intuition - a vector full of numbers of high magnitude will have a higher magnitude than one with numbers of lower magnitude
#dot product is sum of multiplying individual components together
#for maths reasons that i'm not going to explore, the product of the magnitudes is guaranteed to to be higher than the dot product 
#i leave this as an exercise to the reader


#what does rms represent that cosine similarity does not? 
#it shows difference in magnitudes
#high rms and low cosine similarity means that they have nothing in commmon
#low rms and low cosine similarity means that they are about the same magnitude but arranged differently
#high rms and high cosine similarity means that they are the same pattern but at different sizes (i.e, same feature but different activations)
#low rms and high cosine similarity means that they are the same pattern but at 

#for the common neurons of 8 and 0, their activations in their residuals are 0.93 similar
#for the common neurons of 8 and 9, their activations in their residuals are 0.83 similar
#for the common neurons of 8 and 6, their activations in their residuals are 0.81 similar

#conclusion - 8 and 0 are similarly represented in model space

In [12]:
print(similarity_on_set(residuals[0], residuals[9], set0_9))
print(similarity_on_set(residuals[0], residuals[6], set0_6))
print(similarity_on_set(residuals[0], residuals[8], set0_8))

{'cosine_similarity': 0.8129746317863464, 'rms_diff': 0.7617504000663757}
{'cosine_similarity': 0.7407304644584656, 'rms_diff': 1.0948654413223267}
{'cosine_similarity': 0.9296437501907349, 'rms_diff': 0.9510633945465088}


In [13]:
#0 and 6 are represented quite distantly in model space. their vectors point in quite different directions
#the rms is also quite high
#this means that the activations of the shared neurons respectively are in different patterns 
#i.e, their shared neurons are just part of a larger feature or just sloptivations

#0 and 8 are represented quite similarly
#the cosine similarity is high and the rms diff is low
#so their shared neurons are representing the same feature

#out of each digit's residual neurons, 

In [14]:
#are these rms differences actually meaningful? 
#yes
#i'm comparing statistics about activations on common neurons with activations on common neurons 

In [15]:
#next steps
#clean up order of these
#ablate shared 0 and 8 neurons and see what happens
#experiment with different k numbers